# Spherical Neural Surface (SNS) — Normals Demo

This notebook:
1. Loads a pretrained SNS model (MAX10606)
2. Pushes an icosphere through the SNS map
3. Computes surface normals via `BatchDiffQuant`
4. Displays the surface coloured by normals
5. Displays normals as quiver arrows on the surface

## 1. Path setup

In [111]:
import sys, os

_here = os.path.abspath('.')
NS_ROOT = os.path.join(_here, 'neural_surfaces-main')
if not os.path.isdir(NS_ROOT):
    # Already inside neural_surfaces-main from a previous run
    NS_ROOT = _here

if NS_ROOT not in sys.path:
    sys.path.insert(0, NS_ROOT)

os.chdir(NS_ROOT)
print('Working directory:', os.getcwd())

Working directory: /Users/romywilliamson/Documents/SphericalNS/sns-bonn/neural_surfaces-main


## 2. Imports

In [112]:
import torch
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from models.mlp import ResidualMLP
from differential import DifferentialModule, BatchDiffQuant

## 3. Load a pretrained SNS model

Change `SURF_NAME` to `'ARMADILLO21622'` or `'SMALLFLOWER'` for the other available pretrained models.

In [113]:
SURF_NAME = 'MAX10606'
# Architecture matches experiment_configs/overfit/MAX10606.json
model = ResidualMLP({
    'input_size':  3,
    'output_size': 3,
    'layers':      [256, 256, 256, 256, 256, 256, 256, 256],
    'act':         'Softplus',
    'act_params':  {},
    'bias':        False,
    'norm':        None,
    'drop':        0.0,
})

weights_path = os.path.join('..', 'data', 'SNS', SURF_NAME, 'weights.pth')
weights = torch.load(weights_path, map_location='cpu')
model.load_state_dict(weights)
model.eval()

print(f'Loaded SNS model for {SURF_NAME}')

dict_keys(['input_size', 'output_size', 'layers', 'act', 'act_params', 'bias', 'norm', 'drop'])
Res MLP init map weights called
yh 1
Loaded SNS model for MAX10606


## 4. Load an icosphere mesh (parametrisation domain)

Available levels: 0–3, 6, 7  
Higher level = finer mesh. Level 6 gives ~163 k vertices — use level 3 for a quick demo.

In [114]:
SPHERE_LEVEL = 5  # change to 3 for a faster run, 7 for highest detail
sphere_path = os.path.join('..', 'data', 'analytic', 'sphere', f'sphere{SPHERE_LEVEL}.obj')
tm_sphere = trimesh.load(sphere_path)

vertices = np.array(tm_sphere.vertices, dtype=np.float32)
faces    = np.array(tm_sphere.faces)

print(f'Sphere mesh: {vertices.shape[0]:,} vertices, {faces.shape[0]:,} faces')

Sphere mesh: 10,242 vertices, 20,480 faces


## 5. Choose visualisation mode and compute

| `VIS_MODE` | What is shown |
|---|---|
| `'normals'` | Surface normals encoded as RGB |
| `'mean_curvature'` | Mean curvature H (diverging colormap) |
| `'gaussian_curvature'` | Gaussian curvature K (diverging colormap) |
| `'max_curvature'` | Maximum absolute principal curvature |
| `'principal_directions'` | Crossfield: ±dir₁ (blue) and ±dir₂ (red) arrowheads on grey surface |
| `'area_distortion'` | Local area distortion from the sphere |
| `'laplace_beltrami'` | Laplace-Beltrami of a sine scalar field (diverging colormap) |

In [115]:
BATCH_SIZE = 10_000  # reduce if you run out of memory

# ── Visualisation mode ────────────────────────────────────────────────
VIS_MODE = 'principal_directions'   # 'normals' | 'mean_curvature' | 'gaussian_curvature' | 'max_curvature'
                       # 'principal_directions' | 'area_distortion' | 'laplace_beltrami'

# ── Colormap and scaling (applies to scalar modes only) ───────────────
# CMAP: any matplotlib colormap name
#   diverging (good for curvature):  'seismic', 'RdBu_r', 'coolwarm', 'bwr'
#   sequential (good for distortion): 'viridis', 'plasma', 'hot', 'magma'

if VIS_MODE == 'normals' or VIS_MODE == 'principal_directions':
    CMAP = None
elif VIS_MODE in ('mean_curvature', 'max_curvature', 'gaussian_curvature'):
    CMAP = 'seismic'
elif VIS_MODE == 'laplace_beltrami':
    CMAP = 'viridis'
elif VIS_MODE == 'area_distortion':
    CMAP = 'hot'

# SCALING: how raw scalar values are mapped to [0, 1] before the colormap is applied
#   'mapping12'           — sigmoid(0.1 * x)                  (gentle S-curve)
#   'mapping13'           — sigmoid(0.1 * sign(x) * |x|^0.5)  (signed sqrt + sigmoid)
#   'linear5'             — x/20, centred at 0.5
#   'linear9'             — x/5,  centred at 0.5
#   'quadratic'           — sign(x)*sqrt(|x|)/20, centred at 0.5
#   'positive_only_linear1' — 0.05x clipped to [0, 1]  (unsigned)
#   'logmap'              — log-scaled, for area distortion

if VIS_MODE == 'mean_curvature':
    SCALING = 'linear9'
elif VIS_MODE == 'gaussian_curvature':
    SCALING = 'linear5'
elif VIS_MODE == 'area_distortion':
    SCALING = 'logmap'
elif VIS_MODE == 'laplace_beltrami':
    SCALING = 'linear5'

# ── Principal curvature direction (crossfield) settings ───────────────
# Only used when VIS_MODE == 'principal_directions'
PCD_ARROW_LENGTH  = 0.005   # world-space length of each arrowhead
PCD_OFFSET_FACTOR = 0.01    # push arrows off the surface along the normal
PCD_RATIO         = 10.0    # tip length / base width (higher = more pointed)
PCD_SUBSAMPLE     = 5       # use every Nth vertex; increase for fewer arrows

# ── Compute ────────────────────────────────────────────────────────────
diffmod = DifferentialModule()
bdq = BatchDiffQuant(vertices, model, diffmod, batch_size=BATCH_SIZE)

if VIS_MODE == 'normals':
    bdq.compute_normals()
elif VIS_MODE == 'mean_curvature':
    bdq.compute_curvature()
elif VIS_MODE == 'gaussian_curvature':
    bdq.compute_curvature()
elif VIS_MODE in ('max_curvature', 'principal_directions'):
    bdq.compute_directions()
elif VIS_MODE == 'area_distortion':
    bdq.compute_area_distortions()
elif VIS_MODE == 'laplace_beltrami':
    bdq.compute_curvature()
    bdq.compute_normals()

all_output_vertices = bdq.all_output_vertices
print(f'Done — VIS_MODE={VIS_MODE!r}, {all_output_vertices.shape[0]:,} vertices')

Done — VIS_MODE='principal_directions', 10,242 vertices


In [116]:
# Compute Laplace-Beltrami of a sine scalar field — only runs when VIS_MODE == 'laplace_beltrami'
#
# Uses laplace_beltrami_MC:  LB(f) = Δ(f) − 2H (∇f · n) − nᵀ Hess(f) n
# where H and n come from the SNS model, and the analytic derivatives of f
# are supplied explicitly (same approach as visuals/visLaplace.py).

if VIS_MODE == 'laplace_beltrami':
    from torch.nn import functional as F

    # ── Scalar field: f(x) = scale * sin(freq * (x · d))  for a fixed unit direction d ──
    LB_FUNC_FREQ  = 3.0
    LB_FUNC_SCALE = 1.0
    torch.manual_seed(1)
    d = F.normalize(torch.randn(1, 3), dim=1).squeeze()   # fixed unit direction in R3

    surf = torch.tensor(all_output_vertices, dtype=torch.float32)
    arg  = LB_FUNC_FREQ * (surf * d).sum(-1)              # (N,)

    # Analytic value, gradient, and Hessian of f evaluated at every surface vertex
    func_vals = LB_FUNC_SCALE * torch.sin(arg)                                      # (N,)
    grad_vals = LB_FUNC_SCALE * d * (LB_FUNC_FREQ * torch.cos(arg)).unsqueeze(-1)  # (N, 3)
    hess_vals = (
        (d.unsqueeze(-1) @ d.unsqueeze(0)).unsqueeze(0)                              # (1, 3, 3)
        * (LB_FUNC_SCALE * (-LB_FUNC_FREQ**2) * torch.sin(arg))
          .unsqueeze(-1).unsqueeze(-1)                                                # (N, 1, 1)
    )                                                                                 # (N, 3, 3)

    # ── Apply laplace_beltrami_MC ─────────────────────────────────────────────────────
    lb_values = diffmod.laplace_beltrami_MC(
        normals   = torch.tensor(bdq.all_normals, dtype=torch.float32),
        meancurv  = torch.tensor(bdq.all_H,       dtype=torch.float32),
        f         = func_vals,
        grad_f    = grad_vals,
        hessian_f = hess_vals,
    ).detach().numpy()

    print(f'f(x) range:  [{func_vals.min():.3f}, {func_vals.max():.3f}]')
    print(f'LB(f) range: [{lb_values.min():.3f}, {lb_values.max():.3f}]')

## 6. Map to vertex colours

In [117]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from visuals.helpers.colourmappings import (
    mapping12, mapping13,
    linear, linear2, linear3, linear5, linear6, linear7, linear8, linear9,
    quadratic, positive_only_linear1,
    scaled_normals_cmap, scaled_normals_cmap2, logmap
)

_SCALINGS = {
    'mapping12': mapping12, 'mapping13': mapping13,
    'linear': linear, 'linear2': linear2, 'linear3': linear3,
    'linear5': linear5, 'linear6': linear6, 'linear7': linear7,
    'linear8': linear8, 'linear9': linear9,
    'quadratic': quadratic, 'positive_only_linear1': positive_only_linear1,
    'scaled_normals_cmap': scaled_normals_cmap, 'scaled_normals_cmap2': scaled_normals_cmap2,
    'logmap': logmap,
}

def scalar_to_rgb(values, cmap, scaling):
    if scaling == 'percentile':
        lo, hi = np.percentile(values, [2, 98])
        scaled = Normalize(vmin=lo, vmax=hi, clip=True)(values)
    else:
        scaled = np.clip(_SCALINGS[scaling](values), 0.0, 1.0)
    return plt.get_cmap(cmap)(scaled)[:, :3].astype(np.float64)

vertex_rgb = None   # default; not used for principal_directions

if VIS_MODE == 'normals':
    vertex_rgb = np.clip(0.5 * bdq.all_normals + 0.5, 0.0, 1.0)
elif VIS_MODE == 'mean_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_H, CMAP, SCALING)
elif VIS_MODE == 'gaussian_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_K, CMAP, SCALING)
elif VIS_MODE == 'max_curvature':
    vertex_rgb = scalar_to_rgb(bdq.all_MAC, CMAP, SCALING)
elif VIS_MODE == 'area_distortion':
    vertex_rgb = scalar_to_rgb(bdq.all_area_distortions, CMAP, SCALING)
elif VIS_MODE == 'laplace_beltrami':
    func_vertex_rgb = scalar_to_rgb(func_vals.numpy(), CMAP, SCALING)
    vertex_rgb      = scalar_to_rgb(lb_values,          CMAP, SCALING)

if vertex_rgb is not None:
    print(f'vertex_rgb: shape={vertex_rgb.shape}, range=[{vertex_rgb.min():.3f}, {vertex_rgb.max():.3f}]')
else:
    print('principal_directions mode — crossfield viewer launched in next cell')

principal_directions mode — crossfield viewer launched in next cell


In [118]:
import matplotlib.cm as cm
from matplotlib.colors import Normalize, TwoSlopeNorm

def _make_colorbar(ax, values, cmap_name, label):
    """Colorbar with p2–p98 range; symmetric about 0 for diverging cmaps."""
    lo, hi = np.percentile(values, [2, 98])
    diverging = cmap_name in ('seismic', 'RdBu_r', 'coolwarm', 'bwr', 'PRGn',
                               'PiYG', 'PuOr', 'RdGy', 'Spectral')
    if diverging and lo < 0 < hi:
        bound = max(abs(lo), abs(hi))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0, vmax=bound)
    else:
        norm = Normalize(vmin=lo, vmax=hi)
    sm = cm.ScalarMappable(cmap=cmap_name, norm=norm)
    sm.set_array([])
    cb = fig_cb.colorbar(sm, cax=ax)
    cb.set_label(label, fontsize=9)

if VIS_MODE in ('normals', 'principal_directions'):
    print('No colorbar for RGB-encoded modes.')
else:
    _cb_entries = {
        'mean_curvature':    ([bdq.all_H],                          ['mean curvature H']),
        'gaussian_curvature':([bdq.all_K],                          ['Gaussian curvature K']),
        'max_curvature':     ([bdq.all_MAC],                        ['max |principal curvature|']),
        'area_distortion':   ([bdq.all_area_distortions],           ['area distortion']),
        'laplace_beltrami':  ([func_vals.numpy(), lb_values],       ['f(x)', 'LB(f)']),
    }
    arrays, labels = _cb_entries[VIS_MODE]
    n = len(arrays)
    fig_cb, axes_cb = plt.subplots(1, n, figsize=(1.5 * n, 4))
    if n == 1:
        axes_cb = [axes_cb]
    for vals, label, ax in zip(arrays, labels, axes_cb):
        _make_colorbar(ax, vals, CMAP, label)
    fig_cb.suptitle(VIS_MODE.replace('_', ' '), fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

No colorbar for RGB-encoded modes.


## 7. View surface in Open3D

In [119]:
import open3d as o3d
import tempfile, subprocess, sys, os

def _o3d_mesh(V, F, colours):
    m = o3d.geometry.TriangleMesh()
    m.vertices      = o3d.utility.Vector3dVector(np.asarray(V, dtype=np.float64))
    m.triangles     = o3d.utility.Vector3iVector(np.asarray(F))
    m.vertex_colors = o3d.utility.Vector3dVector(np.asarray(colours, dtype=np.float64))
    m.compute_vertex_normals()
    return m

def _launch_viewer(*meshes, window_name):
    tmps = [tempfile.mktemp(suffix='.ply') for _ in meshes]
    for m, t in zip(meshes, tmps):
        o3d.io.write_triangle_mesh(t, m)
    load_lines = '\n'.join(
        f'g{i} = o3d.io.read_triangle_mesh({repr(t)})' for i, t in enumerate(tmps))
    geom_list   = ', '.join(f'g{i}' for i in range(len(tmps)))
    unlink_lines = '\n'.join(f'os.unlink({repr(t)})' for t in tmps)
    script = f"""
import open3d as o3d, os
{load_lines}
vis = o3d.visualization.Visualizer()
vis.create_window(window_name={repr(window_name)}, width=1024, height=768)
for g in [{geom_list}]:
    vis.add_geometry(g)
while vis.poll_events():
    vis.update_renderer()
vis.destroy_window()
{unlink_lines}
"""
    subprocess.Popen([sys.executable, '-c', script])

if VIS_MODE == 'principal_directions':
    # ── Build crossfield arrowhead geometry ───────────────────────────
    # Vectorised replication of writequadsF (visuals/helpers/crossfield.py).
    # Each vertex gets ±dir1 (blue) and ±dir2 (red) arrowheads.
    def _arrow_mesh(pts, dir_main, dir_perp, normals, al, offset, ratio, overlap=True):
        V1    = pts + offset * normals
        N     = len(V1)
        shift = al * dir_main if overlap else np.zeros_like(dir_main)
        V = np.empty((N * 4, 3))
        V[0::4] = V1 - shift
        V[1::4] = V1 + al * dir_main + al * dir_perp - shift
        V[2::4] = V1 + al * dir_main - al * dir_perp - shift
        V[3::4] = V1 + ratio * al * dir_main          - shift
        idx = np.arange(N)
        F = np.empty((N * 2, 3), dtype=np.int32)
        F[0::2] = np.stack([4*idx+1, 4*idx,   4*idx+2], axis=1)
        F[1::2] = np.stack([4*idx+1, 4*idx+2, 4*idx+3], axis=1)
        return V.astype(np.float64), F

    step = PCD_SUBSAMPLE
    pts  = bdq.all_output_vertices[::step]
    d1   = bdq.all_directions[0][::step]
    d2   = bdq.all_directions[1][::step]
    nrm  = bdq.all_normals[::step]
    al, off, rat = PCD_ARROW_LENGTH, PCD_OFFSET_FACTOR, PCD_RATIO

    Va, Fa = _arrow_mesh(pts,  d1, d2, nrm, al, off, rat)
    Vc, Fc = _arrow_mesh(pts, -d1, d2, nrm, al, off, rat)
    V_blue = np.vstack([Va, Vc]);  F_blue = np.vstack([Fa, Fc + len(Va)])

    Vb, Fb = _arrow_mesh(pts,  d2, d1, nrm, al, off, rat)
    Vd, Fd = _arrow_mesh(pts, -d2, d1, nrm, al, off, rat)
    V_red  = np.vstack([Vb, Vd]);  F_red  = np.vstack([Fb, Fd + len(Vb)])

    grey = np.full((len(bdq.all_output_vertices), 3), 0.75)
    mesh_surf = _o3d_mesh(bdq.all_output_vertices, faces, grey)
    mesh_blue = _o3d_mesh(V_blue, F_blue, np.tile([0.0, 0.2, 0.9], (len(V_blue), 1)))
    mesh_red  = _o3d_mesh(V_red,  F_red,  np.tile([0.9, 0.1, 0.1], (len(V_red),  1)))

    _launch_viewer(mesh_surf, mesh_blue, mesh_red,
                   window_name=f'Principal curvature directions — {SURF_NAME}')
    print(f'Crossfield viewer launched — {len(pts):,} sample points on {SURF_NAME}')
    print('  blue = dir₁,  red = dir₂')

else:
    _launch_viewer(_o3d_mesh(all_output_vertices, faces, vertex_rgb),
                   window_name=f'SNS surface ({SURF_NAME}) — {VIS_MODE}')
    print(f'Viewer launched — {VIS_MODE} on {SURF_NAME}')
    if VIS_MODE == 'laplace_beltrami':
        _launch_viewer(_o3d_mesh(all_output_vertices, faces, func_vertex_rgb),
                       window_name=f'SNS surface ({SURF_NAME}) — scalar field f(x)')
        print(f'Viewer launched — scalar field f(x) on {SURF_NAME}')

Crossfield viewer launched — 2,049 sample points on MAX10606
  blue = dir₁,  red = dir₂


## 8. View sphere (parametrisation domain) in Open3D

Same vertex colours as the surface above, so you can see how the colouring maps back to the sphere.

In [ ]:
if VIS_MODE != 'principal_directions':
    import open3d as o3d, tempfile, subprocess, sys, os

    sphere_o3d = _o3d_mesh(vertices, faces, vertex_rgb)
    tmp_ply = tempfile.mktemp(suffix='.ply')
    o3d.io.write_triangle_mesh(tmp_ply, sphere_o3d)

    viewer_script = f"""
import open3d as o3d, os
mesh = o3d.io.read_triangle_mesh({repr(tmp_ply)})
vis = o3d.visualization.Visualizer()
vis.create_window(window_name='Sphere — {SURF_NAME} — {VIS_MODE}', width=1024, height=768)
vis.add_geometry(mesh)
while vis.poll_events():
    vis.update_renderer()
vis.destroy_window()
os.unlink({repr(tmp_ply)})
"""
    subprocess.Popen([sys.executable, '-c', viewer_script])
    print(f'Sphere viewer launched — {VIS_MODE} on {SURF_NAME}')

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
